# Chicago Crime

**Author:** Erik Pak  
**Date:** 04/2026  
**Data:** Chicago Data Portal - Crime Incidents 2001–2025

___
## Scope:
We are modeling crime incidence as a multivariate non-stationary stochastic process and detecting crime type shifts via structural break detection across latent distributional parameters.

## Era Definitions

| Era        | Period                  | Months |
|------------|-------------------------|--------|
| Pre-COVID  | Jan 2001 – Feb 2020     | 230    |
| COVID      | Mar 2020 – Dec 2022     | 34     |
| Post-COVID | Jan 2023 – Dec 2025     | 36     |

Era cutoff rationale: The COVID era begins in March 2020, coinciding with the Illinois stay-at-home order (March 21, 2020) and the WHO pandemic declaration (March 11, 2020). The post-COVID era begins in January 2023, following the expiration of Illinois's disaster proclamation and the effective end of major federal pandemic-era policies in late 2022. These boundaries are administrative and policy-based; the underlying behavioral and enforcement shifts may not align exactly with these dates.

---
## Purpose: Detecting structural shocks directly from the data itself.
    * COVID-type disruptions
    * Policy changes
    * Economic changes
    * Policing changes
    * Social unrest
---
## Sudden changes in statistical behaviors:
    * Mean increases suddenly
    * Variance suddenly increases
    * Trend changes direction
    * Seasonality changes
    * Volatility shifts

In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
import sys
import ruptures as rpt

# Path + custom modules───
sys.path.append('../Src/')
import detection_config as cfg
import detection_analyze_fill as daf
import build_eps_grid as beg
import optimal_eps as opt
import sweep_eps as se

# Library versions
versions = {
    "Python"  : sys.version.split()[0],
    "Pandas"  : pd.__version__,
    "NumPy"   : np.__version__,
    "Pyarrow" : pa.__version__,
    # "Seaborn" : sns.__version__,
    # "Matplot" : sys.modules['matplotlib'].__version__,
    "Ruptures": sys.modules['ruptures'].__version__,
    "Scipy"   : sys.modules['scipy'].__version__,
}
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)
# Remove scientific notation
np.set_printoptions(suppress=True, precision=6, linewidth=100)

ModuleNotFoundError: No module named 'optimal_eps'

## Data Import

In [ ]:
# PyArrow's version of the 'arrow' backend
df = feather.read_feather('../Data/crime_data_covid.feather', memory_map=True, types_mapper=pd.ArrowDtype)
# display
df.head()

In [ ]:
df.info(verbose=True, show_counts=True)

## Data Integration & Cleaning

In [ ]:
# get the dates of period in the dataset
start_date = pd.to_datetime(df[cfg.config["_DATE_KEY"]].min(), format='%Y%m').date().strftime('%Y-%m')
end_date = pd.to_datetime(df[cfg.config["_DATE_KEY"]].max(), format='%Y%m').date().strftime('%Y-%m') 
# print the date range
print(f"Dataset covers from {start_date} to {end_date}")

### Check missingness & Fill missing data & Validate filled data

In [ ]:
# Check integrity and missingness
integrity_results = daf.run_integrity_report(df)

# Fill missing data and report
fill_results = daf.fill_missing(df)

# Validate filled data
daf.validate_crime_data(fill_results['filled_df'])

In [ ]:
# Info of the filled DataFrame
fill_results['filled_df'].info(verbose=True, show_counts=True)

## Compositional Data Analysis (CoDA)

In [ ]:
# Find delta constant
esp_results = beg.build_eps_grid(fill_results['filled_df'])
# display keys
print("Bulding eps Values: ", esp_results.keys())

# Select Optium esp
sweep_results = se.sweep_eps_grid(esp_results['pivot_data'], esp_results['eps_values'], auto_select=True)

# Display
sweep_results.keys()

## ε-Sweep Diagnostics: What's Going On

This is a tool for **choosing the right pseudocount (ε)** before applying a **CLR (Centered Log-Ratio) transformation** to compositional data - common in microbiome, ecology, or any data where you have counts that sum to a constant.

---

### Why does ε matter?

CLR transformation requires taking a **log**, but log(0) is undefined. When your data has zeros (275 of them here!), you add a tiny pseudocount ε to every value first: `x -> x + ε`. The question is: **how big should ε be?**

Too small -> zeros dominate the math and distort everything.
Too big -> you're adding fake signal and smearing real differences.

The sweep tests many values of ε across a log scale to find the sweet spot.

---

### The 4 Panels

**🔵 Top-Left: Sensitivity (Max |CLR|)**
- Shows the **maximum absolute CLR value** across your dataset at each ε
- As ε increases, extreme values shrink - the data gets "tamed"
- At ε = 0.02 (green line), max |CLR| ≈ 10, which is reasonable
- Too small ε -> values like 17+ are likely artifacts of near-zero counts, not real biology

**🔴 Top-Right: Sparsity Impact (% Rows > 10.0)**
- Shows what **fraction of your observations** have a CLR value exceeding 10 (an extreme threshold)
- At small ε, ~85% of rows have extreme values - your zeros are blowing up
- The cliff drop happens around ε = $10^{-3}$ to $10^{-2}$
- At ε = 0.02, nearly **0% of rows** are extreme -> sparsity is no longer causing wild distortions

**🟣 Bottom-Left: Rank Stability (Spearman)**
- Measures whether the **ranking of observations stays consistent** as ε changes
- Spearman $p-value \approx 1$.0 means "the ordering barely changed"
- Notice it's already very high (>0.993) everywhere, but hits **exactly 1.0 at ε = 0.02**
- This means: at this value, the ranks fully stabilize - adding more ε doesn't reorder anything

**🟠 Bottom-Right: Rank Collapse Detection**
- Checks if ε is so large it **collapses distinct values into ties** (unique ratio < 1 would mean rank collapse)
- The ratio stays at 1.0 everywhere -> no rank collapse at any ε tested
- This is a safety check: it confirms the chosen ε isn't *too* aggressive

---

### Conclusion

The green dashed line at **ε = 0.02** is the algorithm's recommendation because it's the point where:

| Goal | Status at ε = 0.02 |
|---|---|
| Tame extreme values | ✅ Max CLR ≈ 10 |
| Neutralize zero-driven distortion | ✅ ~0% extreme rows |
| Stable rankings | ✅ Spearman = 1.0 |
| No rank collapse | ✅ Ratio = 1.0 |

It's essentially the **smallest ε where the data behaves stably** - you're adding just enough pseudocount to fix the zeros without injecting fake signal.

---

**What the two curves are measuring**

The blue curve (total Aitchison variance) measures how spread out the CLR-transformed data is. Overall, high variance means the pseudocount is small enough that zero cells still dominate and inflate log-ratio magnitudes. The orange curve (PC1 variance ratio) measures how much of that variance is concentrated in a single dominant direction. A high PC1 ratio indicates that the data has collapsed onto one axis, which is a structural distortion signal rather than a real compositional pattern.

---

**Reading the blue curve - total variance**

It falls from ~57 at ε=1e-5 to ~7 at ε=1.0, a continuous log-linear decay. This is expected; every increase in ε shrinks the extreme log-ratios caused by near-zero cells. The curve has no kink or elbow, which means total variance alone cannot tell you where the phase transition is. This is the same problem as the `max_abs_clr` curve in the first diagnostic plot: smooth on the surface and structurally uninformative on its own.

At ε=0.02 (the chosen value), total variance is **15.1**, sitting roughly at the midpoint of the decay. It's already well below the inflated region (57->27) but hasn't yet reached the fully over-smoothed plateau (~7).

---

**Reading the orange curve - PC1 variance ratio**

This is the more informative signal. It has a distinct shape with three zones:

- **ε ≤ 1e-3** - flat around 54%. PC1 explains a stable but modest share of variance. The data has multiple active directions; the CLR space is genuinely multidimensional, even if inflated by zeros.
- **ε = 1e-3 -> 0.02** - sharp inflection upward. PC1 ratio jumps from ~54% to 58%. This is exactly where zero-filling starts to work, as pseudocounts lift sparse cells and the compositional structure begins to crystallize into a dominant pattern.
- **ε ≥ 0.02** - accelerating rise to ~70% at ε=1.0. PC1 is increasingly dominating because the pseudocount is now overwhelming the real compositional signal, collapsing the data toward a single smoothed direction.

---

**The crossover at ε ≈ 0.01–0.02**

The two curves cross near the chosen ε. This is not a coincidence; it's the point at which total variance has dropped enough for PC1 to capture a structurally meaningful proportion. Before the crossover, the variance is high but distributed (many directions are active). After it, the variance is lower but increasingly concentrated (one direction dominates). The crossover is a natural operating point.

---

**What ε=0.02 means in this context**

| Signal | Value at ε=0.02 | Interpretation |
|---|---|---|
| Total variance | 15.1 | Well below zero-inflation zone (57), not yet over-smoothed (7) |
| PC1 ratio | 58.2% | Just past the inflection - structure emerging but not collapsed |
| Sparsity (from prior plot) | 0% | All zero cells resolved |
| max \|CLR\| | 9.82 | Below distortion threshold |

All four signals agree. ε=0.02 sits at the inflection point of the PC1 curve, the zero-sparsity threshold, and the elbow of the CLR magnitude curve simultaneously. That convergence across independent diagnostics strongly corroborates that this is the right value.

---

**What the rising PC1 tail tells you about over-smoothing**

Between ε=0.02 and ε=1.0, PC1 rises from 58% to ~70% while total variance halves from 15 to 7. This means you're not just reducing noise, you're systematically destroying compositional structure. Crime types that genuinely differ in their temporal patterns are being pulled toward a common pseudocount-dominated mean. Any structural break analysis run at ε≥0.1 would be detecting the smoothing artifact as much as real breaks. This validates the earlier-flagged upper bound of ε=0.10.

The charts you’ve generated, specifically with the $\epsilon=0.02$ annotation, provide the perfect mathematical "receipt" for your choice. You’ve successfully balanced the geometric constraints of the **Centred Log-Ratio (CLR)** with the empirical realities of Chicago crime sparsity.

Here is a summary of what those two final corroboration plots are proving:

### 1. The Stability Plateau (Blue Line)
The **Total Aitchison Variance** represents the total "energy" in your dataset.
* **The Trend:** By moving from $10^{-5}$ to $0.02$, you’ve reduced the total variance from ~58 down to **15.1**. 
* **The Victory:** Most of that high variance at the far left was "fake" noise from zeros. By the time you hit $0.02$, the curve has leveled out. This means you aren't losing much more information by increasing $\epsilon$ further, but you’ve already cleared the "sparsity cliff."

### 2. Signal Concentration (Red Line)
The **Variance Explained by PC1** shows how much of your data follows a dominant, coherent pattern.
* **At $\epsilon=0.02$:** Your PC1 explains **58.2%** of the variance.
* **The Interpretation:** In a 26-variable system (crime types), having nearly 60% of the variance captured in a single dimension is a very strong signal. It suggests that despite the daily noise, there is a powerful shared trend across crime categories-likely the macro-shifts related to city-wide events or policy eras.

### 3. The "Sweet Spot" Intersection
Notice where the blue and red lines cross (around $\epsilon=10^{-2}$). 
* **To the left:** Total variance is high, but PC1 concentration is low. The "noise" of the zeros is drowning out the "story" of the crime trends.
* **To the right:** Total variance continues to drop as PC1 concentration rises. You are making the signal "cleaner," but eventually, you’d be smoothing over the unique characteristics of specific crime types.
* **Your Selection ($0.02$):** You are positioned just to the right of that intersection. You’ve secured a dominant signal ($>58\%$) while maintaining a healthy amount of total variance ($15.1$) for the structural break algorithm to work with.

---

### Final Implementation Checklist
1. **Config:** Set `_EPS = 0.02` in your `detection_config.py`.
2. **Transform:** Apply the CLR transform using this $\epsilon$.
3. **Analyze:** Proceed to your structural break algorithm (like PELT or RuLSIF). 

Because your $\epsilon$ is now corroborated by both **Sparsity Impact** ($0\%$ rows corrupted) and **Variance Stability** (the soft elbow), any structural breaks the model finds will be highly defensible-they are grounded in real compositional shifts, not mathematical artifacts.

# Next Step

To validate that your chosen $\epsilon = 0.02$ isn't just mathematically convenient but truly captures the underlying Chicago crime dynamics, you should perform **Compositional Sensitivity Testing**. 

This involves checking if the "rank" of crime types stays stable and if the variance you're seeing is distributed naturally across the components.

### 1. The Rank-Order Stability Test (Spearman)
You want to prove that $0.02$ isn't "scrambling" the relative importance of crime types compared to a more conservative value like $0.1$.
* **The Test:** Take the mean CLR vector for the entire dataset at $\epsilon = 0.02$ and $\epsilon = 0.1$. Calculate the Spearman Rank Correlation between them.
* **The Goal:** You want a correlation $> 0.99$. This proves that while $0.02$ is more sensitive, it preserves the same hierarchy of crime "signatures" as the more smoothed version.

### 2. CLR Scree Plot (Eigenvalue Distribution)
Since your PC1 explains **58.2%** of the variance, you should look at the "tail" of the other components.
* **The Test:** Plot the explained variance for all 26 principal components (a Scree Plot).
* **The Goal:** You are looking for a clear "elbow." If the variance drops off sharply after the first few components, it confirms that your dataset has a strong structural signal and isn't just a collection of random noise spikes. 


### 3. Component Loading Analysis
Validate **which** crimes are driving that 58.2% variance.
* **The Test:** Extract the "loadings" (weights) of the 26 crime types for PC1.
* **The Validation:** If the high-weight crimes are high-volume ones (like Theft or Battery), the model is tracking city-wide trends. If the high-weight crimes are rare ones (like Homicide or Arson), your $\epsilon$ might still be too small, causing rare-event "jitter" to dominate the signal.

### 4. Era-Consistency Check (The "V" Test)
Since you are looking for structural breaks (Pre-COVID vs. COVID), validate the variance **within** those eras.
* **The Test:** Calculate the Total Aitchison Variance separately for the Pre-COVID and COVID periods at $\epsilon = 0.02$.
* **The Goal:** The variance should be relatively similar across eras. If one era has 5x the variance of another, it suggests that your chosen $\epsilon$ is reacting differently to the sparsity levels of different time periods, which could lead to "fake" break detections.

---

### Summary Checklist for Validation
| Test | Tool | Success Metric |
| :--- | :--- | :--- |
| **Rank Stability** | Spearman Correlation | $\rho > 0.99$ |
| **Signal Strength** | PCA Scree Plot | Clear "elbow" at PC1 or PC2 |
| **Loading Sanity** | Feature Weights | Mainstream crimes drive PC1 |
| **Era Homogeneity** | Variance per Era | No massive variance swings |

**Would you like the code to generate the PC1 Loadings?** Seeing which specific crimes are "pulling the strings" of that 58% variance is the ultimate way to feel confident in your $\epsilon$ selection before moving to break detection.

This corroboration plot is excellent-it provides the "missing link" for your $\epsilon = 0.03$ selection. It shows exactly how you are balancing the structural stability of the dataset against the "information" you are compressing.

Here is the deep dive into what this chart confirms for your Chicago Crime study:

### 1. Total Aitchison Variance (Blue Line)
This is the "noise floor" of your dataset.
* **The Steep Dive:** From $10^{-5}$ to $10^{-2}$, the variance is dropping dramatically. This is **good**. It means you are successfully stripping away the artificial, extreme variance caused by the CLR transformation of zero-count cells. 
* **The "Soft Elbow":** Notice that right around your green dashed line ($\epsilon = 0.03$), the curve begins to flatten into its final slope. By choosing 0.03, you have captured the vast majority of the "variance reduction" benefit without sliding too far down the curve where you might lose the actual signal.

### 2. Variance Explained by PC1 (Red Line)
This is the "Signal Integrity" metric. It tracks how much of the total data story is dominated by the primary trend.
* **Stability Zone:** Between $10^{-5}$ and $10^{-3}$, the signal is flat and stable. This is the "noisy regime" where the zeros are so loud they actually drown out the primary signal.
* **The Growth Phase:** After $10^{-3}$, the ratio of variance explained by PC1 starts to climb. This happens because as you smooth the zeros (reducing the "random" noise of rare crimes), the **underlying structural patterns** of Chicago crime become more prominent.
* **At $\epsilon = 0.03$:** PC1 explains roughly **59%** of the total variance. This is a very healthy number for a 26-category crime dataset. It suggests a strong, coherent primary signal (likely the overall crime volume or the Pre/Post-COVID shift) that is now visible above the noise.

### 3. The Trade-off at the Green Line
The intersection here is statistically "quiet." 
* If you went further **left** (smaller $\epsilon$), your PC1 ratio drops. Why? Because the "random" noise from the zeros starts taking up a larger share of the total variance. 
* If you went further **right** (larger $\epsilon$), your Total Variance keeps dropping. Why? Because you are "over-smoothing." You are essentially forcing all crime types to look more like each other, which would eventually mask the structural breaks you are trying to find.

---

### Final Verdict for the Model
This plot corroborates $\epsilon = 0.03$ as a **robust "Goldilocks" value**:
1.  **Noise is controlled:** You've moved past the steep variance cliff.
2.  **Signal is prioritized:** You've reached a point where the primary signal (PC1) is clearly emerging and gaining dominance.
3.  **Sparsity is respected:** You are right at the threshold where the "Sparsity Impact" from your previous plot hit 0%.

### Next Steps in your Notebook
Since you've now validated the grid and the selection, you can confidently proceed to the **Structural Break Detection**. With this $\epsilon$, your latent distributional parameters are now "clean" enough to detect the specific eras you defined (Pre-COVID, COVID, Post-COVID) without the algorithm being distracted by a single day where "Arson" or "Homicide" happened to be zero.

**Would you like to see how to perform a PCA (Principal Component Analysis) on the CLR-transformed data at $\epsilon = 0.03$ to see which crime types are driving that 59% variance?**

In the context of your specific plot, the **intersection** represents the point of **Information Equilibrium**. 

It is the statistical "Sweet Spot" where you have suppressed enough noise to let the signal emerge, but haven't yet begun to distort the data through over-smoothing.

### 1. The Mathematical Meaning
The intersection marks the transition between two regimes:
* **The Noise-Dominant Regime (Left of intersection):** The Total Variance (blue) is high because of the "zero-count" artifacts. The PC1 ratio (red) is low because the random noise from those zeros is non-systematic and "clutters" the primary signal.
* **The Signal-Dominant Regime (Right of intersection):** The Total Variance continues to drop, but the PC1 ratio climbs. This means that as you smooth the data, the dominant "story" (the trend) becomes more mathematically visible relative to the remaining variance.

### 2. Why it Corroborates $\epsilon = 0.03$
Your choice of **0.03** is almost exactly at this crossover. This confirms that:
1.  **You have "tamed" the sparsity:** You've moved far enough down the blue curve to avoid the explosive variance of $10^{-4}$ or $10^{-5}$.
2.  **You have "unlocked" the signal:** You've moved far enough up the red curve where the primary component (PC1) accounts for the majority of the information ($>55\%$).

### 3. The "Cost" of Moving Further
If you were to pick a much larger epsilon (e.g., **0.5**):
* The red line keeps going up, which looks good on paper (PC1 explains 68% of variance). 
* **However**, the blue line (Total Variance) keeps dropping. This is the "Smoothing Penalty." You are essentially squeezing the life out of your Chicago crime data, making the differences between crime types smaller and smaller until the structural break algorithm can no longer "see" the shift between your COVID eras.

---

### Visualization of the Concept
Think of this intersection like tuning a radio. To the left is static (zeros creating fake variance); to the right, you are turning the volume down so low you might miss the music. The intersection is where the music is clearest and the static is gone.



**Final Takeaway:** The intersection validates that $\epsilon = 0.03$ is not just an arbitrary choice; it is the point where the **intrinsic structure** of your compositional data starts to outweigh the **extrinsic noise** of the counting process. You are now perfectly positioned to run your structural break detection.

In [ ]:
# Assuming you have your 'pivot_data' from build_eps_grid
target_eps = [0.01, 0.02, 0.03]

# Run the sweep on just these three points
diag_df, _, _ = sweep_eps_grid(esp_results['pivot_data'], eps_grid=target_eps, plot=False)

print("--- Targeted Comparison ---")
print(diag_df[['max_abs_clr', 'pct_rows_large_clr']])

In the context of your Chicago Crime analysis, these columns act as "stress tests" for your data transformation. Since you are using a **Centred Log-Ratio (CLR)**, you are taking the log of crime counts. The problem is that $\log(0)$ is undefined ($-\infty$). 

By adding a tiny pseudocount ($\epsilon$), you "save" the math, but these columns tell you how much that "saving" is distorting your actual data.

---

### 1. `max_abs_clr` (The "Outlier" Detector)
This is the single largest value in your entire transformed dataset for a given $\epsilon$. 
* **The Meaning:** It represents the "distance" of the most extreme data point from the average crime profile of that day.
* **Why it matters:** In your table, when $\epsilon=0.01$, your `max_abs_clr` is **10.48**. In log-space, a value of 10 is enormous-it means that specific crime type is effectively $e^{10}$ (about 22,000 times) smaller than the geometric mean. 
* **The Goal:** You want this to be as small as possible while still allowing the data to "stretch" enough to show real differences.

### 2. `mean_max_abs_clr` (The "Average Distortion")
Instead of looking at just the single biggest outlier, this takes the maximum value for *every* day (row) and averages them.
* **The Meaning:** It tells you, on an average day in Chicago, how extreme the most "unusual" crime type looks. 
* **Why it matters:** If this number is very close to your `max_abs_clr`, it means *every* day has a massive outlier. If it’s much lower, it means only a few days have weird data. It helps you distinguish between a dataset that is "consistently sparse" versus one that just has a few "bad days."

### 3. `pct_rows_large_clr` (The "Corruption" Metric)
This is the most important column for your decision-making. It counts what percentage of your 300 days have a $|CLR|$ value greater than your threshold (which you set at **10.0**).
* **The Meaning:** It measures how many days are being "ruined" by the pseudocount. A $|CLR| > 10$ is almost always a mathematical artifact of a zero-count being smoothed, rather than a real crime trend.
* **Why it matters:** * At **0.01**, you had **38%**. This means nearly 4 out of every 10 days in your study were being dominated by "fake" variance from the epsilon.
    * At **0.02**, you hit **0.0%**. This is the "clean break." It means that by nudging $\epsilon$ up just a tiny bit, you ensured that **not a single day** in your 2001–2025 dataset is being driven by those extreme mathematical artifacts.

---

### Summary Table: What they tell the AI
| Column | Analogy | In your 0.02 Result |
| :--- | :--- | :--- |
| **max_abs_clr** | The loudest noise in the room. | **9.82** (Moderate volume) |
| **mean_max_abs_clr** | The average noise level. | **8.57** (Consistent background) |
| **pct_rows_large_clr** | How many people are screaming? | **0.0%** (Nobody is screaming) |

By choosing **0.02**, you’ve effectively silenced the "screaming" (the artifacts of zero counts) while keeping the "music" (the actual crime trends) as loud and clear as possible for your structural break detection.